# Volume 9, Chapter 3: Noetherian Rings and Hilbert's Theorems (Python companion)

Buchberger's algorithm written from scratch, solving polynomial systems by elimination, and colouring graphs and solving Shidoku by ideal membership.

*Before each solution, predict what the output will look like.*

## Exercises

**P3.1 (★★) Buchberger's algorithm from scratch.** Implement the division algorithm of Proposition 3.15 and Buchberger's algorithm (Corollary 3.20) with polynomials stored as dictionaries `{exponent tuple: Fraction}`, for the lexicographic and graded reverse lexicographic orders. Reduce the result (make leading coefficients $1$, drop elements whose leading monomial is divisible by another's, and reduce each element by the others) and compare with `sympy.groebner` on several systems, including the one of Exercise 3.6. Count the S-polynomials computed. Which order gives smaller bases?

**P3.2 (★) Solving a system with a lexicographic basis.** For the intersection of the circle $x^2+y^2=4$, the hyperbola $xy=1$ and the cubic surface $x^3+y^3+z^3=1$ in $\C^3$, compute a lexicographic Gröbner basis, check that its last element involves only $z$, and find all the solutions numerically by back substitution. How many are there, and how many are real?

**P3.3 (★★) CS application: colouring graphs and solving Shidoku by ideal membership.** (a) With the encoding of Exercise 3.5, decide with Gröbner bases whether the 5-cycle, the complete graph $K_4$ and the wheel with $5$ spokes are $3$-colourable, by testing whether the reduced Gröbner basis is $\{1\}$. (b) Shidoku is Sudoku on a $4\times4$ grid with $2\times2$ boxes. Encode the cell values $1,2,3,4$ as the roots of $F(x)=(x-1)(x-2)(x-3)(x-4)$ and the constraint "cells $u\ne v$" by the polynomial $\frac{F(x_u)-F(x_v)}{x_u-x_v}$. Solve a puzzle with a few givens by computing a lexicographic Gröbner basis, and use the dimension of the quotient ring (the number of standard monomials) to count its solutions.

## Solutions

### P3.1

In [ ]:
from fractions import Fraction
from itertools import combinations
import sympy as sp

def lex_key(a):  return a
def grevlex_key(a):  return (sum(a), tuple(-x for x in reversed(a)))

class Ring:
    def __init__(self, n, order):
        self.n, self.key = n, {"lex": lex_key, "grevlex": grevlex_key}[order]
    def lm(self, f):  return max(f, key=self.key)
    def add(self, f, g, c=1):
        h = dict(f)
        for m, v in g.items():
            h[m] = h.get(m, 0) + c * v
            if h[m] == 0: del h[m]
        return h
    def mulmono(self, f, mono, c):
        return {tuple(a + b for a, b in zip(m, mono)): c * v for m, v in f.items()}
    def divides(self, a, b):  return all(x <= y for x, y in zip(a, b))

    def divide(self, f, G):
        """Proposition 3.15: remainder of f on division by the list G."""
        p, r = dict(f), {}
        while p:
            m = self.lm(p)
            for g in G:
                lg = self.lm(g)
                if self.divides(lg, m):
                    q = tuple(x - y for x, y in zip(m, lg))
                    p = self.add(p, self.mulmono(g, q, p[m] / g[lg]), -1)
                    break
            else:
                r[m] = p.pop(m)
        return r

    def spoly(self, f, g):
        a, b = self.lm(f), self.lm(g)
        gam = tuple(max(x, y) for x, y in zip(a, b))
        return self.add(self.mulmono(f, tuple(x - y for x, y in zip(gam, a)), 1 / f[a]),
                        self.mulmono(g, tuple(x - y for x, y in zip(gam, b)), 1 / g[b]), -1)

    def buchberger(self, F):
        G = [dict(f) for f in F]
        pairs, count = list(combinations(range(len(G)), 2)), 0
        while pairs:
            i, j = pairs.pop()
            count += 1
            r = self.divide(self.spoly(G[i], G[j]), G)
            if r:
                G.append(r)
                pairs += [(k, len(G) - 1) for k in range(len(G) - 1)]
        return self.reduce(G), count

    def reduce(self, G):
        G = [{m: v / g[self.lm(g)] for m, v in g.items()} for g in G]
        keep = []
        for i, g in enumerate(G):         # drop g if its leading monomial is divisible by another's
            if not any(self.divides(self.lm(h), self.lm(g)) and (self.lm(h) != self.lm(g) or j < i)
                       for j, h in enumerate(G) if j != i):
                keep.append(g)
        return sorted([self.divide(g, [h for h in keep if h is not g]) for g in keep],
                      key=lambda g: self.key(self.lm(g)), reverse=True)

def from_sympy(expr, gens):
    P = sp.Poly(expr, *gens)
    return {m: Fraction(int(c.p), int(c.q)) for m, c in P.terms()}

def to_sympy(f, gens):
    return sp.expand(sum(sp.Rational(v.numerator, v.denominator) * sp.prod([g**e for g, e in zip(gens, m)]) for m, v in f.items()))

x, y, z = sp.symbols("x y z")
systems = {
    "Exercise 3.6": ([x**2 - y, x**3 - x], (x, y)),
    "twisted cubic": ([x**2 - y, x**3 - z], (x, y, z)),
    "circle & hyperbola": ([x**2 + y**2 - 4, x*y - 1], (x, y)),
    "three quadrics": ([x**2 + y*z - 2, y**2 + x*z - 2, z**2 + x*y - 2], (x, y, z)),
}
for name, (F, gens) in systems.items():
    for order in ("lex", "grevlex"):
        R = Ring(len(gens), order)
        G, count = R.buchberger([from_sympy(f, gens) for f in F])
        ours = sorted(sp.srepr(sp.expand(to_sympy(g, gens))) for g in G)
        theirs = sorted(sp.srepr(sp.expand(g)) for g in sp.groebner(F, *gens, order=order).exprs)
        print(f"{name:20s} {order:8s} {len(G)} elements, {count:3d} S-polynomials, same as sympy: {ours == theirs}")
    print("   lex basis:", [to_sympy(g, gens) for g in Ring(len(gens), "lex").buchberger([from_sympy(f, gens) for f in F])[0]])

Our reduced bases coincide with sympy's (a reduced Gröbner basis is unique for a given order). The lexicographic bases are triangular, which makes them useful for solving, but they can be far more expensive: $378$ S-polynomials against $15$ for the three quadrics, even though the final lexicographic basis is not always the larger one. The graded reverse lexicographic order is the usual choice when one only wants to decide membership or compute dimensions.

### P3.2

In [ ]:
import numpy as np
F = [x**2 + y**2 - 4, x*y - 1, x**3 + y**3 + z**3 - 1]
G = sp.groebner(F, x, y, z, order="lex")
for g in G.exprs:
    print(sp.factor(g))

With $x>y>z$ the lexicographic basis eliminates $x$ first: the last element involves only $z$, the ones before it only $y$ and $z$, and the first ones $x$. Solve from the bottom up:

In [ ]:
sols = []
gz = [g for g in G.exprs if g.free_symbols <= {z}][0]
for zr in sp.Poly(gz, z).nroots(n=30):
    gy = [sp.Poly(g.subs(z, zr), y) for g in G.exprs if g.free_symbols <= {y, z} and y in g.free_symbols]
    for yr in gy[0].nroots(n=30):
        if all(abs(complex(p.eval(yr))) < 1e-12 for p in gy[1:]):
            gx = [sp.Poly(g.subs({z: zr, y: yr}), x) for g in G.exprs if x in g.free_symbols]
            for xr in gx[0].nroots(n=30):
                if all(abs(complex(p.eval(xr))) < 1e-12 for p in gx):
                    sols.append((complex(xr), complex(yr), complex(zr)))
sols = list({tuple(np.round(s, 8)) for s in sols})
real = [s for s in sols if all(abs(c.imag) < 1e-9 for c in s)]
print(len(sols), "solutions in C^3, of which", len(real), "real:")
for s in sorted(real, key=lambda s: s[0].real):
    print("  ", tuple(round(float(c.real), 6) for c in s))
print("max residual:", max(abs(complex(f.subs({x: a, y: b, z: c}))) for (a, b, c) in sols for f in F))

The circle and the hyperbola meet in $4$ points of the plane, all real ($x^2+y^2=4$, $xy=1$ give $(x+y)^2=6$, $(x-y)^2=2$), and above each of them $z^3=1-x^3-y^3$ has $3$ complex roots, one real: $12$ solutions, $4$ real. Bézout's bound $2\cdot2\cdot3=12$ is attained.

### P3.3

In [ ]:
def colouring_ideal(n, edges):
    X = sp.symbols(f"x0:{n}")
    F = [X[v]**3 - 1 for v in range(n)] + [X[u]**2 + X[u]*X[v] + X[v]**2 for u, v in edges]
    return F, X

graphs = {
    "5-cycle": (5, [(i, (i + 1) % 5) for i in range(5)]),
    "K4": (4, list(combinations(range(4), 2))),
    "wheel W5": (6, [(i, (i + 1) % 5) for i in range(5)] + [(5, i) for i in range(5)]),
}
for name, (n, E) in graphs.items():
    F, X = colouring_ideal(n, E)
    G = sp.groebner(F, *X, order="grevlex")
    print(f"{name:9s}: reduced Groebner basis is {{1}}: {G.exprs == [1]}  ->  3-colourable: {G.exprs != [1]}")

The odd cycle needs three colours and has them; $K_4$ needs four; the wheel with an odd rim needs four as well, since the hub uses one colour and the odd rim needs three more. In each non-colourable case, the Gröbner computation produces a certificate $1=\sum a_ig_i$, a proof of impossibility that can be checked by expanding polynomials.

In [ ]:
grid = [[1, 0, 0, 0],
        [0, 0, 1, 0],
        [0, 4, 0, 0],
        [0, 0, 0, 0]]           # 0 = empty
cells = [(r, c) for r in range(4) for c in range(4)]
X = {rc: sp.Symbol(f"c{rc[0]}{rc[1]}") for rc in cells}
t, u = sp.symbols("t u")
Fpol = sp.expand((t - 1) * (t - 2) * (t - 3) * (t - 4))
diff = sp.Poly(sp.cancel((Fpol - Fpol.subs(t, u)) / (t - u)), t, u)

def neq(a, b):
    return diff.as_expr().subs({t: X[a], u: X[b]}, simultaneous=True)

def same_unit(a, b):
    return a[0] == b[0] or a[1] == b[1] or (a[0] // 2, a[1] // 2) == (b[0] // 2, b[1] // 2)

eqs = [Fpol.subs(t, X[a]) for a in cells]
eqs += [neq(a, b) for a, b in combinations(cells, 2) if same_unit(a, b)]
eqs += [X[a] - grid[a[0]][a[1]] for a in cells if grid[a[0]][a[1]]]
gens = [X[a] for a in cells]
G = sp.groebner(eqs, *gens, order="lex")
print(len(G.exprs), "polynomials in the reduced lex basis; the last ones:", G.exprs[-3:])

The givens fix some cells at once (their basis elements are linear), and the cells that are not determined uniquely appear in elements of higher degree. Count the solutions by the dimension of the quotient ring, which is the number of monomials divisible by no leading monomial, and read them off:

In [ ]:
lead = [sp.Poly(g, *gens).monoms(order="lex")[0] for g in G.exprs]
def standard_monomials(lead, n):
    """Monomials not divisible by any leading monomial; finite for a zero-dimensional ideal."""
    out, frontier = [], [tuple([0] * n)]
    seen = set(frontier)
    while frontier:
        m = frontier.pop()
        if any(all(a <= b for a, b in zip(l, m)) for l in lead):
            continue
        out.append(m)
        for i in range(n):
            m2 = tuple(m[j] + (j == i) for j in range(n))
            if m2 not in seen:
                seen.add(m2); frontier.append(m2)
    return out
std = standard_monomials(lead, len(gens))
print("dimension of the quotient = number of solutions:", len(std))
for s in sp.solve(G.exprs, gens, dict=True):
    print(np.array([[s.get(X[(r, c)], X[(r, c)]) for c in range(4)] for r in range(4)]))

The ideal is radical (each cell polynomial $F$ has distinct roots), so by the Nullstellensatz the dimension of $\C[x]/I$ is the number of points of $V(I)$: the number of completions of the puzzle. With these three givens the puzzle has $6$ completions; adding a $3$ in the bottom right corner makes the solution unique (try it), and contradictory clues make the basis collapse to $\{1\}$. With no givens at all, the same computation counts the $288$ Shidoku grids. A $9\times9$ Sudoku can be encoded in the same way, but the Gröbner computation becomes heavy; constraint propagation is much faster there, and the algebraic method shines rather when it produces certificates of impossibility.